In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(
    "../data/processed/clean_superstore.csv",
    parse_dates=["order_date", "ship_date"]
)

## Create date features

In [ ]:
#Order year: Allows you to compare sales across years.
df["order_year"] = df["order_date"].dt.year

In [ ]:
#Order month: Useful for identifying seasonality.
df["order_month"] = df["order_date"].dt.month_name()

In [ ]:
#Month number: Without this, months sort alphabetically instead of chronologically.
df["month_number"] = df["order_date"].dt.month

In [9]:
#Quarter: Many businesses report quarterly performance.
df["quarter"] = df["order_date"].dt.quarter

In [10]:
#Day of Week: Useful if you want to see whether weekends or weekdays have different sales patterns.
df["day_of_week"] = df["order_date"].dt.day_name()

## Shipping Features

Shipping Days

Business Question: How long does it take to deliver an order?

In [11]:
df["shipping_days"] = (
    df["ship_date"] - df["order_date"]
).dt.days

In [12]:
df["shipping_days"].describe()

count    9994.000000
mean        3.958175
std         1.747567
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: shipping_days, dtype: float64

## Profitability Features

Profit Margin (%)

Business Question: How much profit is earned for every dollar of sales?

In [13]:
df["profit_margin"] = (
    df["profit"] / df["sales"]
) * 100

## Discount Categories

In [14]:
#Instead of raw discount values, group them into business-friendly categories.
bins = [-0.01, 0, 0.2, 0.5, 1]
labels = ["No Discount", "Low", "Medium", "High"]

df["discount_level"] = pd.cut(
    df["discount"],
    bins=bins,
    labels=labels
)

In [15]:
df["discount_level"].value_counts()

discount_level
No Discount    4798
Low            3803
High            856
Medium          537
Name: count, dtype: int64

## Sales Categories 

In [16]:
#Categorize orders by sales amount.
bins = [0, 100, 500, 1000, df["sales"].max()]
labels = ["Low", "Medium", "High", "Very High"]

df["sales_category"] = pd.cut(
    df["sales"],
    bins=bins,
    labels=labels
)

## Order Size

In [17]:
def order_size(quantity):
    if quantity <= 2:
        return "Small"
    elif quantity <= 5:
        return "Medium"
    else:
        return "Large"

df["order_size"] = df["quantity"].apply(order_size)

## Validate New Features

In [18]:
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,order_year,order_month,month_number,quarter,day_of_week,shipping_days,profit_margin,discount_level,sales_category,order_size
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,2016,November,11,4,Tuesday,3,16.00,No Discount,Medium,Small
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,2016,November,11,4,Tuesday,3,30.00,No Discount,High,Medium
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,2016,June,6,2,Sunday,4,47.00,No Discount,Low,Small
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,2015,October,10,4,Sunday,7,-40.00,Medium,High,Medium
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,2015,October,10,4,Sunday,7,11.25,Low,Low,Small


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 31 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   row_id          9994 non-null   int64         
 1   order_id        9994 non-null   object        
 2   order_date      9994 non-null   datetime64[ns]
 3   ship_date       9994 non-null   datetime64[ns]
 4   ship_mode       9994 non-null   object        
 5   customer_id     9994 non-null   object        
 6   customer_name   9994 non-null   object        
 7   segment         9994 non-null   object        
 8   country         9994 non-null   object        
 9   city            9994 non-null   object        
 10  state           9994 non-null   object        
 11  postal_code     9994 non-null   int64         
 12  region          9994 non-null   object        
 13  product_id      9994 non-null   object        
 14  category        9994 non-null   object        
 15  sub_

In [20]:
df.describe()

,row_id,order_date,ship_date,postal_code,sales,quantity,discount,profit,order_year,month_number,quarter,shipping_days,profit_margin
count,9994.000000,9994,9994,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,2016-04-30 00:07:12.259355648,2016-05-03 23:06:58.571142912,55190.379428,229.858001,3.789574,0.156203,28.656896,2015.722233,7.809686,2.882329,3.958175,12.031393
min,1.000000,2014-01-03 00:00:00,2014-01-07 00:00:00,1040.000000,0.444000,1.000000,0.000000,-6599.978000,2014.000000,1.000000,1.000000,0.000000,-275.000000
25%,2499.250000,2015-05-23 00:00:00,2015-05-27 00:00:00,23223.000000,17.280000,2.000000,0.000000,1.728750,2015.000000,5.000000,2.000000,3.000000,7.500000
50%,4997.500000,2016-06-26 00:00:00,2016-06-29 00:00:00,56430.500000,54.490000,3.000000,0.200000,8.666500,2016.000000,9.000000,3.000000,4.000000,27.000000
75%,7495.750000,2017-05-14 00:00:00,2017-05-18 00:00:00,90008.000000,209.940000,5.000000,0.200000,29.364000,2017.000000,11.000000,4.000000,5.000000,36.250000
max,9994.000000,2017-12-30 00:00:00,2018-01-05 00:00:00,99301.000000,22638.480000,14.000000,0.800000,8399.976000,2017.000000,12.000000,4.000000,7.000000,50.000000
std,2885.163629,NaN,NaN,32063.693350,623.245101,2.225110,0.206452,234.260108,1.123555,3.284654,1.058086,1.747567,46.675435


Save the Feature-Engineered Dataset

In [21]:
df.to_csv(
    "../data/processed/superstore_features.csv",
    index=False
)